In [1]:
from pathlib import Path
import pandas as pd

# Load the aggregated sales data using an absolute path.
DATA_PATH = Path(
    r"D:\ai-fmcg-sales-insights\data\processed\daily_store_dept_sales.csv"
)

df = pd.read_csv(DATA_PATH)

# Standardize column names and parse dates.
df.columns = df.columns.str.strip().str.lower()
df["sales_date"] = pd.to_datetime(df["sales_date"], errors="raise")

# Sort each time series chronologically.
df = df.sort_values(
    ["store_id", "dept_id", "sales_date"]
).reset_index(drop=True)

# Check scope and data quality.
series_counts = df.groupby(["store_id", "dept_id"]).size()

print("Rows:", len(df))
print("Date range:", df["sales_date"].min(), "to", df["sales_date"].max())
print("Stores:", df["store_id"].nunique())
print("Departments:", df["dept_id"].nunique())
print("Time series:", len(series_counts))
print("Rows per series:", series_counts.min(), "to", series_counts.max())
print("Missing values:")
print(df.isna().sum())
print(
    "Duplicate keys:",
    df.duplicated(["store_id", "dept_id", "sales_date"]).sum()
)
print("Negative units:", (df["total_units_sold"] < 0).sum())
print("Total units:", df["total_units_sold"].sum())
print("Total revenue:", round(df["total_sales_revenue"].sum(), 2))

display(df.head())

Rows: 21900
Date range: 2014-05-24 00:00:00 to 2016-05-22 00:00:00
Stores: 10
Departments: 3
Time series: 30
Rows per series: 730 to 730
Missing values:
sales_date             0
state_id               0
store_id               0
dept_id                0
total_units_sold       0
total_sales_revenue    0
dtype: int64
Duplicate keys: 0
Negative units: 0
Total units: 18481172
Total revenue: 47686404.71


,sales_date,state_id,store_id,dept_id,total_units_sold,total_sales_revenue
0,2014-05-24,CA,CA_1,FOODS_1,343,750.20
1,2014-05-25,CA,CA_1,FOODS_1,379,683.36
2,2014-05-26,CA,CA_1,FOODS_1,250,586.27
3,2014-05-27,CA,CA_1,FOODS_1,207,442.35
4,2014-05-28,CA,CA_1,FOODS_1,197,502.52


In [2]:
# Reserve the final two 28-day periods for validation and testing.
test_start = df["sales_date"].max() - pd.Timedelta(days=27)
validation_start = test_start - pd.Timedelta(days=28)

train_df = df[df["sales_date"] < validation_start].copy()

validation_df = df[
    (df["sales_date"] >= validation_start)
    & (df["sales_date"] < test_start)
].copy()

test_df = df[df["sales_date"] >= test_start].copy()

# Verify the size and date range of each split.
for name, split in [
    ("Train", train_df),
    ("Validation", validation_df),
    ("Test", test_df),
]:
    print(
        f"{name}: {len(split):,} rows | "
        f"{split['sales_date'].min().date()} to "
        f"{split['sales_date'].max().date()} | "
        f"{split['sales_date'].nunique()} days"
    )

assert len(train_df) + len(validation_df) + len(test_df) == len(df)
assert train_df["sales_date"].max() < validation_df["sales_date"].min()
assert validation_df["sales_date"].max() < test_df["sales_date"].min()

Train: 20,220 rows | 2014-05-24 to 2016-03-27 | 674 days
Validation: 840 rows | 2016-03-28 to 2016-04-24 | 28 days
Test: 840 rows | 2016-04-25 to 2016-05-22 | 28 days


In [3]:
# Use the final training week as a repeating weekly forecast.
forecast_start = validation_df["sales_date"].min()

last_week = train_df[
    train_df["sales_date"] >= forecast_start - pd.Timedelta(days=7)
][["store_id", "dept_id", "sales_date", "total_units_sold"]].copy()

last_week["day_of_week"] = last_week["sales_date"].dt.dayofweek
last_week = last_week.rename(
    columns={"total_units_sold": "baseline_prediction"}
)

assert len(last_week) == 30 * 7
assert not last_week.duplicated(
    ["store_id", "dept_id", "day_of_week"]
).any()

# Match future dates to the last observed weekday values.
validation_predictions = validation_df.copy()
validation_predictions["day_of_week"] = (
    validation_predictions["sales_date"].dt.dayofweek
)

validation_predictions = validation_predictions.merge(
    last_week[
        ["store_id", "dept_id", "day_of_week", "baseline_prediction"]
    ],
    on=["store_id", "dept_id", "day_of_week"],
    how="left",
    validate="many_to_one"
)

assert len(validation_predictions) == 840
assert validation_predictions["baseline_prediction"].notna().all()

display(
    validation_predictions[
        ["sales_date", "store_id", "dept_id",
         "total_units_sold", "baseline_prediction"]
    ].head(10)
)

,sales_date,store_id,dept_id,total_units_sold,baseline_prediction
0,2016-03-28,CA_1,FOODS_1,233,295
1,2016-03-29,CA_1,FOODS_1,185,358
2,2016-03-30,CA_1,FOODS_1,215,348
3,2016-03-31,CA_1,FOODS_1,354,335
4,2016-04-01,CA_1,FOODS_1,320,483
5,2016-04-02,CA_1,FOODS_1,311,612
6,2016-04-03,CA_1,FOODS_1,367,411
7,2016-04-04,CA_1,FOODS_1,238,295
8,2016-04-05,CA_1,FOODS_1,246,358
9,2016-04-06,CA_1,FOODS_1,214,348


In [4]:
import numpy as np

# Evaluate predictions across all store-department-day records.
actual = validation_predictions["total_units_sold"].to_numpy(dtype=float)
predicted = validation_predictions["baseline_prediction"].to_numpy(dtype=float)

errors = predicted - actual

mae = np.mean(np.abs(errors))
rmse = np.sqrt(np.mean(errors ** 2))
wape = np.sum(np.abs(errors)) / np.sum(actual) * 100
bias = np.sum(errors) / np.sum(actual) * 100

baseline_metrics = pd.DataFrame({
    "Model": ["Weekly seasonal naive"],
    "MAE": [mae],
    "RMSE": [rmse],
    "WAPE (%)": [wape],
    "Bias (%)": [bias]
})

display(baseline_metrics.round(2))

,Model,MAE,RMSE,WAPE (%),Bias (%)
0,Weekly seasonal naive,152.45,230.33,16.19,-1.1


In [5]:
# Average each weekday over the final four training weeks.
recent_history = train_df[
    train_df["sales_date"] >= forecast_start - pd.Timedelta(days=28)
].copy()

recent_history["day_of_week"] = recent_history["sales_date"].dt.dayofweek

weekday_averages = (
    recent_history
    .groupby(["store_id", "dept_id", "day_of_week"])["total_units_sold"]
    .mean()
    .rename("weekday_average_prediction")
    .reset_index()
)

validation_predictions = validation_predictions.merge(
    weekday_averages,
    on=["store_id", "dept_id", "day_of_week"],
    how="left",
    validate="many_to_one"
)

assert validation_predictions["weekday_average_prediction"].notna().all()

# Evaluate the second baseline using the same validation records.
predicted_avg = validation_predictions[
    "weekday_average_prediction"
].to_numpy(dtype=float)

errors_avg = predicted_avg - actual

average_metrics = pd.DataFrame({
    "Model": ["Four-week weekday average"],
    "MAE": [np.mean(np.abs(errors_avg))],
    "RMSE": [np.sqrt(np.mean(errors_avg ** 2))],
    "WAPE (%)": [np.sum(np.abs(errors_avg)) / np.sum(actual) * 100],
    "Bias (%)": [np.sum(errors_avg) / np.sum(actual) * 100]
})

display(pd.concat(
    [baseline_metrics, average_metrics],
    ignore_index=True
).round(2))

,Model,MAE,RMSE,WAPE (%),Bias (%)
0,Weekly seasonal naive,152.45,230.33,16.19,-1.1
1,Four-week weekday average,98.15,151.79,10.42,-0.0


In [6]:
# Check whether the modeling packages are available.
import importlib.util

for package in ["sklearn", "xgboost"]:
    status = (
        "Installed"
        if importlib.util.find_spec(package) is not None
        else "Not installed"
    )
    print(f"{package}: {status}")

sklearn: Installed
xgboost: Installed


In [7]:
%pip install xgboost

Note: you may need to restart the kernel to use updated packages.


In [8]:
import xgboost as xgb

print("XGBoost version:", xgb.__version__)

XGBoost version: 2.1.4


In [9]:
# Create features using training data only.
train_features = train_df.sort_values(
    ["store_id", "dept_id", "sales_date"]
).copy()

series_keys = ["store_id", "dept_id"]
grouped_units = train_features.groupby(series_keys)["total_units_sold"]

# Add historical sales features.
for lag in [1, 7, 14, 28]:
    train_features[f"lag_{lag}"] = grouped_units.shift(lag)

# Shift before rolling to exclude the target day's sales.
for window in [7, 28]:
    train_features[f"rolling_mean_{window}"] = (
        grouped_units.transform(
            lambda values: values.shift(1).rolling(window).mean()
        )
    )

# Add calendar features known before forecasting.
train_features["day_of_week"] = train_features["sales_date"].dt.dayofweek
train_features["month"] = train_features["sales_date"].dt.month
train_features["day_of_month"] = train_features["sales_date"].dt.day
train_features["time_index"] = (
    train_features["sales_date"] - df["sales_date"].min()
).dt.days

# Remove the initial rows without sufficient history.
history_features = [
    "lag_1", "lag_7", "lag_14", "lag_28",
    "rolling_mean_7", "rolling_mean_28"
]

train_features = train_features.dropna(
    subset=history_features
).reset_index(drop=True)

print("Training rows with features:", len(train_features))

display(
    train_features[
        ["sales_date", "store_id", "dept_id", "total_units_sold"]
        + history_features
    ].head()
)

Training rows with features: 19380


,sales_date,store_id,dept_id,total_units_sold,lag_1,lag_7,lag_14,lag_28,rolling_mean_7,rolling_mean_28
0,2014-06-21,CA_1,FOODS_1,328,382.0,452.0,456.0,343.0,327.428571,340.142857
1,2014-06-22,CA_1,FOODS_1,315,328.0,364.0,405.0,379.0,309.714286,339.607143
2,2014-06-23,CA_1,FOODS_1,405,315.0,304.0,391.0,250.0,302.714286,337.321429
3,2014-06-24,CA_1,FOODS_1,316,405.0,224.0,304.0,207.0,317.142857,342.857143
4,2014-06-25,CA_1,FOODS_1,261,316.0,322.0,340.0,197.0,330.285714,346.750000


In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor

# Define model inputs and the prediction target.
categorical_features = ["store_id", "dept_id"]

numeric_features = history_features + [
    "day_of_week", "month", "day_of_month", "time_index"
]

feature_columns = categorical_features + numeric_features

X_train = train_features[feature_columns]
y_train = train_features["total_units_sold"]

# Encode identifiers without implying a numeric order.
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categories",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        ("numbers", "passthrough", numeric_features)
    ]
)

# Train a reproducible initial model.
xgb_model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", XGBRegressor(
        objective="reg:squarederror",
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=4
    ))
])

xgb_model.fit(X_train, y_train)

print("XGBoost training completed.")
print("Training rows:", len(X_train))
print("Input features:", feature_columns)

XGBoost training completed.
Training rows: 19380
Input features: ['store_id', 'dept_id', 'lag_1', 'lag_7', 'lag_14', 'lag_28', 'rolling_mean_7', 'rolling_mean_28', 'day_of_week', 'month', 'day_of_month', 'time_index']


In [11]:
# Forecast recursively using only observed training history.
def recursive_forecast(model, history_df, future_dates):
    histories = {
        key: group.sort_values("sales_date")["total_units_sold"]
        .astype(float).tolist()
        for key, group in history_df.groupby(["store_id", "dept_id"])
    }

    dates = pd.DatetimeIndex(sorted(future_dates))
    assert dates[0] == history_df["sales_date"].max() + pd.Timedelta(days=1)
    assert (dates.to_series().diff().dropna() == pd.Timedelta(days=1)).all()
    assert all(len(values) >= 28 for values in histories.values())

    results = []

    for date in dates:
        rows = []

        # Build each series' features from its available history.
        for (store, department), values in histories.items():
            row = {
                "store_id": store,
                "dept_id": department,
                "day_of_week": date.dayofweek,
                "month": date.month,
                "day_of_month": date.day,
                "time_index": (date - df["sales_date"].min()).days,
                "rolling_mean_7": np.mean(values[-7:]),
                "rolling_mean_28": np.mean(values[-28:])
            }

            for lag in [1, 7, 14, 28]:
                row[f"lag_{lag}"] = values[-lag]

            rows.append(row)

        inputs = pd.DataFrame(rows)
        predictions = np.maximum(
            model.predict(inputs[feature_columns]), 0
        )
        assert np.isfinite(predictions).all()

        # Update history with predictions, never future actual sales.
        for row, prediction in zip(rows, predictions):
            key = (row["store_id"], row["dept_id"])
            histories[key].append(float(prediction))

            results.append({
                "sales_date": date,
                "store_id": key[0],
                "dept_id": key[1],
                "xgb_prediction": float(prediction)
            })

    return pd.DataFrame(results)


xgb_validation_forecast = recursive_forecast(
    xgb_model,
    train_df,
    validation_df["sales_date"].unique()
)

assert len(xgb_validation_forecast) == 840

print("Forecast rows:", len(xgb_validation_forecast))
display(xgb_validation_forecast.head(10))

Forecast rows: 840


,sales_date,store_id,dept_id,xgb_prediction
0,2016-03-28,CA_1,FOODS_1,341.328522
1,2016-03-28,CA_1,FOODS_2,420.422241
2,2016-03-28,CA_1,FOODS_3,1879.286987
3,2016-03-28,CA_2,FOODS_1,418.890564
4,2016-03-28,CA_2,FOODS_2,474.807922
5,2016-03-28,CA_2,FOODS_3,1674.830933
6,2016-03-28,CA_3,FOODS_1,418.791809
7,2016-03-28,CA_3,FOODS_2,613.769226
8,2016-03-28,CA_3,FOODS_3,3041.847412
9,2016-03-28,CA_4,FOODS_1,250.773117


In [12]:
# Match predictions to actuals using explicit keys.
xgb_validation_results = validation_df.merge(
    xgb_validation_forecast,
    on=["sales_date", "store_id", "dept_id"],
    how="left",
    validate="one_to_one"
)

assert len(xgb_validation_results) == 840
assert xgb_validation_results["xgb_prediction"].notna().all()

# Evaluate XGBoost on the same validation period as the baselines.
xgb_actual = xgb_validation_results[
    "total_units_sold"
].to_numpy(dtype=float)

xgb_predicted = xgb_validation_results[
    "xgb_prediction"
].to_numpy(dtype=float)

xgb_errors = xgb_predicted - xgb_actual

xgb_metrics = pd.DataFrame({
    "Model": ["XGBoost recursive"],
    "MAE": [np.mean(np.abs(xgb_errors))],
    "RMSE": [np.sqrt(np.mean(xgb_errors ** 2))],
    "WAPE (%)": [
        np.sum(np.abs(xgb_errors)) / np.sum(xgb_actual) * 100
    ],
    "Bias (%)": [
        np.sum(xgb_errors) / np.sum(xgb_actual) * 100
    ]
})

validation_comparison = pd.concat(
    [baseline_metrics, average_metrics, xgb_metrics],
    ignore_index=True
).sort_values("WAPE (%)")

display(validation_comparison.round(2))

,Model,MAE,RMSE,WAPE (%),Bias (%)
1,Four-week weekday average,98.15,151.79,10.42,-0.00
2,XGBoost recursive,127.71,171.84,13.56,9.59
0,Weekly seasonal naive,152.45,230.33,16.19,-1.10


In [13]:
# Inspect validation errors by department.
department_results = xgb_validation_results.copy()

department_results["absolute_error"] = (
    department_results["xgb_prediction"]
    - department_results["total_units_sold"]
).abs()

department_summary = department_results.groupby("dept_id").agg(
    actual_units=("total_units_sold", "sum"),
    predicted_units=("xgb_prediction", "sum"),
    absolute_error=("absolute_error", "sum")
)

department_summary["WAPE (%)"] = (
    department_summary["absolute_error"]
    / department_summary["actual_units"] * 100
)

department_summary["Bias (%)"] = (
    (department_summary["predicted_units"]
     - department_summary["actual_units"])
    / department_summary["actual_units"] * 100
)

display(department_summary.round(2))

,actual_units,predicted_units,absolute_error,WAPE (%),Bias (%)
dept_id,,,,,
FOODS_1,81681,110478.69,29251.26,35.81,35.26
FOODS_2,163797,167682.08,21003.46,12.82,2.37
FOODS_3,545454,588643.43,57017.56,10.45,7.92


In [14]:
from sklearn.base import clone
from sklearn.compose import TransformedTargetRegressor

# Train on log-transformed sales and return predictions in original units.
xgb_log_model = TransformedTargetRegressor(
    regressor=clone(xgb_model),
    func=np.log1p,
    inverse_func=np.expm1
)

xgb_log_model.fit(X_train, y_train)

# Use the same recursive 28-day validation procedure.
log_forecast = recursive_forecast(
    xgb_log_model,
    train_df,
    validation_df["sales_date"].unique()
)

log_results = validation_df.merge(
    log_forecast,
    on=["sales_date", "store_id", "dept_id"],
    how="left",
    validate="one_to_one"
)

assert len(log_results) == 840
assert log_results["xgb_prediction"].notna().all()

log_actual = log_results["total_units_sold"].to_numpy(dtype=float)
log_errors = (
    log_results["xgb_prediction"].to_numpy(dtype=float) - log_actual
)

log_metrics = pd.DataFrame({
    "Model": ["XGBoost log target"],
    "MAE": [np.mean(np.abs(log_errors))],
    "RMSE": [np.sqrt(np.mean(log_errors ** 2))],
    "WAPE (%)": [np.sum(np.abs(log_errors)) / np.sum(log_actual) * 100],
    "Bias (%)": [np.sum(log_errors) / np.sum(log_actual) * 100]
})

display(pd.concat(
    [validation_comparison, log_metrics],
    ignore_index=True
).sort_values("WAPE (%)").round(2))

,Model,MAE,RMSE,WAPE (%),Bias (%)
0,Four-week weekday average,98.15,151.79,10.42,-0.00
1,XGBoost recursive,127.71,171.84,13.56,9.59
2,Weekly seasonal naive,152.45,230.33,16.19,-1.10
3,XGBoost log target,245.53,353.14,26.08,24.01


In [15]:
# Use all observations available before the test period.
pretest_history = pd.concat(
    [train_df, validation_df], ignore_index=True
)

test_forecast_start = test_df["sales_date"].min()

recent_pretest = pretest_history[
    pretest_history["sales_date"]
    >= test_forecast_start - pd.Timedelta(days=28)
].copy()

recent_pretest["day_of_week"] = recent_pretest["sales_date"].dt.dayofweek

# Keep the selected four-week averaging rule unchanged.
test_weekday_averages = (
    recent_pretest
    .groupby(["store_id", "dept_id", "day_of_week"])["total_units_sold"]
    .mean()
    .rename("selected_prediction")
    .reset_index()
)

test_results = test_df.copy()
test_results["day_of_week"] = test_results["sales_date"].dt.dayofweek

test_results = test_results.merge(
    test_weekday_averages,
    on=["store_id", "dept_id", "day_of_week"],
    how="left",
    validate="many_to_one"
)

assert len(test_results) == 840
assert test_results["selected_prediction"].notna().all()

# Evaluate the held-out test period.
test_actual = test_results["total_units_sold"].to_numpy(dtype=float)
test_errors = (
    test_results["selected_prediction"].to_numpy(dtype=float)
    - test_actual
)

test_metrics = pd.DataFrame({
    "Model": ["Four-week weekday average"],
    "Split": ["Test"],
    "MAE": [np.mean(np.abs(test_errors))],
    "RMSE": [np.sqrt(np.mean(test_errors ** 2))],
    "WAPE (%)": [np.sum(np.abs(test_errors)) / np.sum(test_actual) * 100],
    "Bias (%)": [np.sum(test_errors) / np.sum(test_actual) * 100]
})

display(test_metrics.round(2))

,Model,Split,MAE,RMSE,WAPE (%),Bias (%)
0,Four-week weekday average,Test,124.09,183.2,12.52,-4.99


In [16]:
# Estimate weekday demand from the final 28 observed days.
last_date = df["sales_date"].max()

latest_history = df[
    df["sales_date"] >= last_date - pd.Timedelta(days=27)
].copy()

latest_history["day_of_week"] = latest_history["sales_date"].dt.dayofweek

future_averages = (
    latest_history
    .groupby(["store_id", "dept_id", "day_of_week"])["total_units_sold"]
    .mean()
    .rename("predicted_units")
    .reset_index()
)

# Create 28 future dates for every store-department series.
future_dates = pd.DataFrame({
    "sales_date": pd.date_range(
        last_date + pd.Timedelta(days=1), periods=28
    )
})

series = df[["state_id", "store_id", "dept_id"]].drop_duplicates()

future_forecast = series.merge(future_dates, how="cross")
future_forecast["day_of_week"] = future_forecast["sales_date"].dt.dayofweek

future_forecast = future_forecast.merge(
    future_averages,
    on=["store_id", "dept_id", "day_of_week"],
    how="left",
    validate="many_to_one"
).sort_values(["store_id", "dept_id", "sales_date"]).reset_index(drop=True)

assert len(future_forecast) == 840
assert future_forecast["predicted_units"].notna().all()

print("Forecast rows:", len(future_forecast))
print(
    "Forecast period:",
    future_forecast["sales_date"].min().date(),
    "to",
    future_forecast["sales_date"].max().date()
)

display(future_forecast.head())

Forecast rows: 840
Forecast period: 2016-05-23 to 2016-06-19


,state_id,store_id,dept_id,sales_date,day_of_week,predicted_units
0,CA,CA_1,FOODS_1,2016-05-23,0,282.75
1,CA,CA_1,FOODS_1,2016-05-24,1,304.75
2,CA,CA_1,FOODS_1,2016-05-25,2,304.00
3,CA,CA_1,FOODS_1,2016-05-26,3,306.25
4,CA,CA_1,FOODS_1,2016-05-27,4,383.25


In [17]:
# Load calendar features, including both event fields.
CALENDAR_PATH = Path(
    r"D:\ai-fmcg-sales-insights\data\raw\m5\calendar.csv"
)

calendar_features = pd.read_csv(
    CALENDAR_PATH,
    usecols=[
        "date",
        "event_name_1", "event_type_1",
        "event_name_2", "event_type_2",
        "snap_CA", "snap_TX", "snap_WI"
    ],
    dtype={
        "event_name_1": "string",
        "event_type_1": "string",
        "event_name_2": "string",
        "event_type_2": "string"
    },
    parse_dates=["date"]
).rename(columns={"date": "sales_date"})

assert not calendar_features["sales_date"].duplicated().any()

# Join calendar information without changing the sales grain.
df_enriched = df.merge(
    calendar_features,
    on="sales_date",
    how="left",
    validate="many_to_one"
)

# Select the SNAP flag corresponding to each store's state.
df_enriched["snap_eligible"] = np.nan

for state in ["CA", "TX", "WI"]:
    mask = df_enriched["state_id"].eq(state)
    df_enriched.loc[mask, "snap_eligible"] = (
        df_enriched.loc[mask, f"snap_{state}"]
    )

assert df_enriched["snap_eligible"].notna().all()
assert df_enriched["snap_eligible"].isin([0, 1]).all()
df_enriched["snap_eligible"] = df_enriched["snap_eligible"].astype(int)

# Treat missing event names as days without a recorded event.
event_columns = [
    "event_name_1", "event_type_1",
    "event_name_2", "event_type_2"
]

df_enriched[event_columns] = (
    df_enriched[event_columns].fillna("None")
)

df_enriched["is_event_day"] = (
    df_enriched["event_name_1"].ne("None")
    | df_enriched["event_name_2"].ne("None")
).astype(int)

assert len(df_enriched) == len(df)
assert df_enriched["total_units_sold"].sum() == df["total_units_sold"].sum()

print("Rows:", len(df_enriched))
print("SNAP values:", sorted(df_enriched["snap_eligible"].unique()))
print(
    "Dates with recorded events:",
    df_enriched.loc[
        df_enriched["is_event_day"].eq(1), "sales_date"
    ].nunique()
)

display(
    df_enriched[
        ["sales_date", "store_id", "state_id",
         "event_name_1", "event_name_2",
         "is_event_day", "snap_eligible"]
    ].head(10)
)

Rows: 21900
SNAP values: [0, 1]
Dates with recorded events: 59


,sales_date,store_id,state_id,event_name_1,event_name_2,is_event_day,snap_eligible
0,2014-05-24,CA_1,CA,None,None,0,0
1,2014-05-25,CA_1,CA,None,None,0,0
2,2014-05-26,CA_1,CA,MemorialDay,None,1,0
3,2014-05-27,CA_1,CA,None,None,0,0
4,2014-05-28,CA_1,CA,None,None,0,0
5,2014-05-29,CA_1,CA,None,None,0,0
6,2014-05-30,CA_1,CA,None,None,0,0
7,2014-05-31,CA_1,CA,None,None,0,0
8,2014-06-01,CA_1,CA,None,None,0,1
9,2014-06-02,CA_1,CA,None,None,0,1


In [18]:
# Attach calendar features to the existing training feature rows.
calendar_model_columns = event_columns + [
    "is_event_day", "snap_eligible"
]

enriched_train_features = train_features.merge(
    df_enriched[
        ["sales_date", "store_id", "dept_id"] + calendar_model_columns
    ],
    on=["sales_date", "store_id", "dept_id"],
    how="left",
    validate="one_to_one"
)

assert len(enriched_train_features) == 19380
assert enriched_train_features[calendar_model_columns].notna().all().all()

# Add event categories and numerical calendar flags.
enriched_categorical_features = categorical_features + event_columns
enriched_numeric_features = numeric_features + [
    "is_event_day", "snap_eligible"
]

enriched_feature_columns = (
    enriched_categorical_features + enriched_numeric_features
)

enriched_preprocessor = ColumnTransformer(
    transformers=[
        (
            "categories",
            OneHotEncoder(handle_unknown="ignore"),
            enriched_categorical_features
        ),
        ("numbers", "passthrough", enriched_numeric_features)
    ]
)

# Keep the original XGBoost settings to compare the added features.
xgb_calendar_model = Pipeline([
    ("preprocessor", enriched_preprocessor),
    ("regressor", clone(xgb_model.named_steps["regressor"]))
])

xgb_calendar_model.fit(
    enriched_train_features[enriched_feature_columns],
    enriched_train_features["total_units_sold"]
)

print("Calendar-enhanced XGBoost training completed.")
print("Training rows:", len(enriched_train_features))

Calendar-enhanced XGBoost training completed.
Training rows: 19380


In [19]:
# Forecast with known calendar information and recursive sales history.
def recursive_calendar_forecast(model, history_df, future_calendar):
    histories = {
        key: group.sort_values("sales_date")["total_units_sold"]
        .astype(float).tolist()
        for key, group in history_df.groupby(["store_id", "dept_id"])
    }

    dates = sorted(future_calendar["sales_date"].unique())
    assert pd.Timestamp(dates[0]) == (
        history_df["sales_date"].max() + pd.Timedelta(days=1)
    )

    results = []

    for date_value in dates:
        date = pd.Timestamp(date_value)
        day_calendar = future_calendar[
            future_calendar["sales_date"].eq(date)
        ]

        rows = []

        for _, calendar_row in day_calendar.iterrows():
            key = (calendar_row["store_id"], calendar_row["dept_id"])
            values = histories[key]

            row = {
                "store_id": key[0],
                "dept_id": key[1],
                "day_of_week": date.dayofweek,
                "month": date.month,
                "day_of_month": date.day,
                "time_index": (date - df["sales_date"].min()).days,
                "rolling_mean_7": np.mean(values[-7:]),
                "rolling_mean_28": np.mean(values[-28:])
            }

            for lag in [1, 7, 14, 28]:
                row[f"lag_{lag}"] = values[-lag]

            for column in calendar_model_columns:
                row[column] = calendar_row[column]

            rows.append(row)

        inputs = pd.DataFrame(rows)
        predictions = np.maximum(
            model.predict(inputs[enriched_feature_columns]), 0
        )
        assert np.isfinite(predictions).all()

        # Append predictions rather than future actual sales.
        for row, prediction in zip(rows, predictions):
            key = (row["store_id"], row["dept_id"])
            histories[key].append(float(prediction))

            results.append({
                "sales_date": date,
                "store_id": key[0],
                "dept_id": key[1],
                "calendar_prediction": float(prediction)
            })

    return pd.DataFrame(results)


# Pass only identifiers and calendar inputs, excluding actual sales.
validation_calendar = df_enriched.loc[
    (df_enriched["sales_date"] >= validation_start)
    & (df_enriched["sales_date"] < test_start),
    ["sales_date", "store_id", "dept_id"] + calendar_model_columns
].copy()

calendar_forecast = recursive_calendar_forecast(
    xgb_calendar_model, train_df, validation_calendar
)

calendar_results = validation_df.merge(
    calendar_forecast,
    on=["sales_date", "store_id", "dept_id"],
    how="left",
    validate="one_to_one"
)

assert len(calendar_results) == 840
assert calendar_results["calendar_prediction"].notna().all()

# Compare validation errors with the original models.
calendar_actual = calendar_results["total_units_sold"].to_numpy(dtype=float)
calendar_errors = (
    calendar_results["calendar_prediction"].to_numpy(dtype=float)
    - calendar_actual
)

calendar_metrics = pd.DataFrame({
    "Model": ["XGBoost + events + SNAP"],
    "MAE": [np.mean(np.abs(calendar_errors))],
    "RMSE": [np.sqrt(np.mean(calendar_errors ** 2))],
    "WAPE (%)": [
        np.sum(np.abs(calendar_errors)) / np.sum(calendar_actual) * 100
    ],
    "Bias (%)": [
        np.sum(calendar_errors) / np.sum(calendar_actual) * 100
    ]
})

display(pd.concat(
    [validation_comparison, calendar_metrics],
    ignore_index=True
).sort_values("WAPE (%)").round(2))

,Model,MAE,RMSE,WAPE (%),Bias (%)
0,Four-week weekday average,98.15,151.79,10.42,-0.00
1,XGBoost recursive,127.71,171.84,13.56,9.59
2,Weekly seasonal naive,152.45,230.33,16.19,-1.10
3,XGBoost + events + SNAP,277.23,354.87,29.44,28.74


In [20]:
from sklearn.linear_model import Ridge

# Estimate calendar adjustments from training history only.
adjustment_train = df_enriched[
    df_enriched["sales_date"] < validation_start
].sort_values(["store_id", "dept_id", "sales_date"]).copy()

# Build a historical baseline from the previous four matching weekdays.
sales_groups = adjustment_train.groupby(
    ["store_id", "dept_id"]
)["total_units_sold"]

weekday_lags = pd.concat(
    [sales_groups.shift(lag) for lag in [7, 14, 21, 28]],
    axis=1
)

adjustment_train["historical_baseline"] = weekday_lags.mean(
    axis=1, skipna=False
)

adjustment_train = adjustment_train.dropna(
    subset=["historical_baseline"]
).copy()

# Learn proportional deviations from the historical baseline.
adjustment_train["relative_deviation"] = (
    adjustment_train["total_units_sold"]
    / adjustment_train["historical_baseline"].clip(lower=1)
    - 1
)

# Include state-specific SNAP indicators.
snap_adjustment_columns = []

for state in ["CA", "TX", "WI"]:
    column = f"snap_flag_{state}"
    snap_adjustment_columns.append(column)
    adjustment_train[column] = (
        adjustment_train["state_id"].eq(state)
        * adjustment_train["snap_eligible"]
    ).astype(int)

adjustment_categories = ["event_type_1", "event_type_2"]
adjustment_columns = adjustment_categories + snap_adjustment_columns

adjustment_preprocessor = ColumnTransformer([
    (
        "events",
        OneHotEncoder(handle_unknown="ignore"),
        adjustment_categories
    ),
    ("snap", "passthrough", snap_adjustment_columns)
])

# Regularization limits large adjustments from sparse event observations.
calendar_adjustment_model = Pipeline([
    ("preprocessor", adjustment_preprocessor),
    ("regressor", Ridge(alpha=100.0, solver="lsqr"))
])

calendar_adjustment_model.fit(
    adjustment_train[adjustment_columns],
    adjustment_train["relative_deviation"]
)

print("Calendar adjustment model fitted.")
print("Training rows:", len(adjustment_train))

Calendar adjustment model fitted.
Training rows: 19380


In [21]:
# Attach calendar information to the original baseline predictions.
adjusted_validation = validation_predictions.merge(
    df_enriched[
        ["sales_date", "store_id", "dept_id"]
        + ["event_type_1", "event_type_2", "snap_eligible"]
    ],
    on=["sales_date", "store_id", "dept_id"],
    how="left",
    validate="one_to_one"
)

# Create the same state-specific SNAP inputs used during fitting.
for state in ["CA", "TX", "WI"]:
    adjusted_validation[f"snap_flag_{state}"] = (
        adjusted_validation["state_id"].eq(state)
        * adjusted_validation["snap_eligible"]
    ).astype(int)

# Apply estimated proportional adjustments.
adjustment = calendar_adjustment_model.predict(
    adjusted_validation[adjustment_columns]
)

adjusted_validation["adjustment_factor"] = np.maximum(
    1 + adjustment, 0
)

adjusted_validation["adjusted_prediction"] = (
    adjusted_validation["weekday_average_prediction"]
    * adjusted_validation["adjustment_factor"]
)

assert len(adjusted_validation) == 840
assert np.isfinite(adjusted_validation["adjusted_prediction"]).all()

# Evaluate on the same validation records.
adjusted_actual = adjusted_validation[
    "total_units_sold"
].to_numpy(dtype=float)

adjusted_errors = (
    adjusted_validation["adjusted_prediction"].to_numpy(dtype=float)
    - adjusted_actual
)

adjusted_metrics = pd.DataFrame({
    "Model": ["Four-week average + calendar adjustment"],
    "MAE": [np.mean(np.abs(adjusted_errors))],
    "RMSE": [np.sqrt(np.mean(adjusted_errors ** 2))],
    "WAPE (%)": [
        np.sum(np.abs(adjusted_errors)) / np.sum(adjusted_actual) * 100
    ],
    "Bias (%)": [
        np.sum(adjusted_errors) / np.sum(adjusted_actual) * 100
    ]
})

display(pd.concat(
    [average_metrics, adjusted_metrics],
    ignore_index=True
).round(2))

,Model,MAE,RMSE,WAPE (%),Bias (%)
0,Four-week weekday average,98.15,151.79,10.42,-0.00
1,Four-week average + calendar adjustment,87.09,129.03,9.25,2.38


In [22]:
from pathlib import Path

# Define and create the output directory.
OUTPUT_PATH = Path(r"D:\ai-fmcg-sales-insights\data\processed")
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

# Save the calendar-adjusted validation results.
adjusted_metrics.assign(Split="Validation").to_csv(
    OUTPUT_PATH / "calendar_adjusted_validation_metrics.csv",
    index=False
)

adjusted_validation.to_csv(
    OUTPUT_PATH / "calendar_adjusted_validation_predictions.csv",
    index=False
)

print("Calendar-adjusted validation results saved.")

Calendar-adjusted validation results saved.


In [23]:
# Define four historical validation windows before the test period
backtest_starts = [
    validation_start - pd.Timedelta(days=offset)
    for offset in [84, 56, 28, 0]
]

backtest_windows = pd.DataFrame({
    "forecast_start": backtest_starts,
    "forecast_end": [
        start + pd.Timedelta(days=27)
        for start in backtest_starts
    ]
})

print(backtest_windows.to_string(index=False))

# Ensure all windows end before the test period
assert backtest_windows["forecast_end"].max() < test_start

forecast_start forecast_end
    2016-01-04   2016-01-31
    2016-02-01   2016-02-28
    2016-02-29   2016-03-27
    2016-03-28   2016-04-24


In [24]:
import numpy as np
from sklearn.base import clone

backtest_records = []

for forecast_start in backtest_starts:
    forecast_end = forecast_start + pd.Timedelta(days=27)

    # Select history available at the forecast origin
    history = df_enriched.loc[
        df_enriched["sales_date"] < forecast_start
    ].sort_values(
        ["store_id", "dept_id", "sales_date"]
    ).copy()

    evaluation = df_enriched.loc[
        df_enriched["sales_date"].between(
            forecast_start, forecast_end
        )
    ].copy()

    assert len(evaluation) == 840
    assert history["sales_date"].max() < forecast_start

    # Build historical baseline features without future sales
    grouped_sales = history.groupby(
        ["store_id", "dept_id"]
    )["total_units_sold"]

    lag_values = pd.concat(
        [grouped_sales.shift(lag) for lag in [7, 14, 21, 28]],
        axis=1
    )

    history["historical_baseline"] = lag_values.mean(
        axis=1, skipna=False
    )

    fitting_data = history.dropna(
        subset=["historical_baseline"]
    ).copy()

    fitting_data["relative_deviation"] = (
        fitting_data["total_units_sold"]
        / fitting_data["historical_baseline"].clip(lower=1)
        - 1
    )

    # Create state-specific SNAP features
    for state in ["CA", "TX", "WI"]:
        column = f"snap_flag_{state}"

        fitting_data[column] = (
            fitting_data["state_id"].eq(state)
            & fitting_data["snap_eligible"].eq(1)
        ).astype(int)

        evaluation[column] = (
            evaluation["state_id"].eq(state)
            & evaluation["snap_eligible"].eq(1)
        ).astype(int)

    # Fit a fresh adjustment model for this window
    fold_model = clone(calendar_adjustment_model)

    fold_model.fit(
        fitting_data[adjustment_columns],
        fitting_data["relative_deviation"]
    )

    # Forecast all 28 days using the latest four observed weeks
    recent_history = history.loc[
        history["sales_date"]
        >= forecast_start - pd.Timedelta(days=28)
    ].copy()

    recent_history["day_of_week"] = (
        recent_history["sales_date"].dt.dayofweek
    )

    weekday_means = (
        recent_history.groupby(
            ["store_id", "dept_id", "day_of_week"],
            as_index=False
        )["total_units_sold"]
        .mean()
        .rename(columns={"total_units_sold": "base_prediction"})
    )

    evaluation["day_of_week"] = (
        evaluation["sales_date"].dt.dayofweek
    )

    evaluation = evaluation.merge(
        weekday_means,
        on=["store_id", "dept_id", "day_of_week"],
        how="left",
        validate="many_to_one"
    )

    adjustment = fold_model.predict(
        evaluation[adjustment_columns]
    )

    evaluation["adjusted_prediction"] = (
        evaluation["base_prediction"]
        * np.maximum(1 + adjustment, 0)
    )

    # Calculate comparable metrics for both models
    actual = evaluation["total_units_sold"].to_numpy(dtype=float)

    for model_name, prediction_column in [
        ("Four-week weekday average", "base_prediction"),
        ("Calendar-adjusted average", "adjusted_prediction")
    ]:
        predicted = evaluation[prediction_column].to_numpy(
            dtype=float
        )

        assert np.isfinite(predicted).all()
        assert actual.sum() > 0

        errors = predicted - actual

        backtest_records.append({
            "Forecast Start": forecast_start.date(),
            "Forecast End": forecast_end.date(),
            "Model": model_name,
            "WAPE (%)": 100 * np.abs(errors).sum() / actual.sum(),
            "Bias (%)": 100 * errors.sum() / actual.sum()
        })

backtest_results = pd.DataFrame(backtest_records)

print(backtest_results.round(2).to_string(index=False))

Forecast Start Forecast End                     Model  WAPE (%)  Bias (%)
    2016-01-04   2016-01-31 Four-week weekday average     18.97    -10.71
    2016-01-04   2016-01-31 Calendar-adjusted average     18.66    -10.69
    2016-02-01   2016-02-28 Four-week weekday average     13.52     -6.59
    2016-02-01   2016-02-28 Calendar-adjusted average     13.04     -5.29
    2016-02-29   2016-03-27 Four-week weekday average     11.30     -1.30
    2016-02-29   2016-03-27 Calendar-adjusted average     11.10      0.62
    2016-03-28   2016-04-24 Four-week weekday average     10.42     -0.00
    2016-03-28   2016-04-24 Calendar-adjusted average      9.25      2.38


In [25]:
# Save rolling backtest results
backtest_results.to_csv(
    OUTPUT_PATH / "forecast_rolling_backtest_results.csv",
    index=False
)

print("Rolling backtest results saved.")

Rolling backtest results saved.


In [26]:
from sklearn.base import clone

# Prepare history strictly before the test period
test_history = df_enriched.loc[
    df_enriched["sales_date"] < test_start
].sort_values(
    ["store_id", "dept_id", "sales_date"]
).copy()

adjusted_test_results = df_enriched.loc[
    df_enriched["sales_date"].between(
        test_start, test_start + pd.Timedelta(days=27)
    )
].copy()

# Build historical baseline features
grouped_sales = test_history.groupby(
    ["store_id", "dept_id"]
)["total_units_sold"]

lag_values = pd.concat(
    [grouped_sales.shift(lag) for lag in [7, 14, 21, 28]],
    axis=1
)

test_history["historical_baseline"] = lag_values.mean(
    axis=1, skipna=False
)

test_fitting_data = test_history.dropna(
    subset=["historical_baseline"]
).copy()

test_fitting_data["relative_deviation"] = (
    test_fitting_data["total_units_sold"]
    / test_fitting_data["historical_baseline"].clip(lower=1)
    - 1
)

# Create state-specific SNAP features
for state in ["CA", "TX", "WI"]:
    column = f"snap_flag_{state}"

    for frame in [test_fitting_data, adjusted_test_results]:
        frame[column] = (
            frame["state_id"].eq(state)
            & frame["snap_eligible"].eq(1)
        ).astype(int)

# Fit using pre-test history only
test_adjustment_model = clone(calendar_adjustment_model)

test_adjustment_model.fit(
    test_fitting_data[adjustment_columns],
    test_fitting_data["relative_deviation"]
)

# Calculate the fixed four-week weekday baseline
recent_history = test_history.loc[
    test_history["sales_date"]
    >= test_start - pd.Timedelta(days=28)
].copy()

recent_history["day_of_week"] = (
    recent_history["sales_date"].dt.dayofweek
)

test_weekday_means = (
    recent_history.groupby(
        ["store_id", "dept_id", "day_of_week"],
        as_index=False
    )["total_units_sold"]
    .mean()
    .rename(columns={"total_units_sold": "base_prediction"})
)

adjusted_test_results["day_of_week"] = (
    adjusted_test_results["sales_date"].dt.dayofweek
)

adjusted_test_results = adjusted_test_results.merge(
    test_weekday_means,
    on=["store_id", "dept_id", "day_of_week"],
    how="left",
    validate="many_to_one"
)

# Apply calendar adjustments without using test sales
adjustment = test_adjustment_model.predict(
    adjusted_test_results[adjustment_columns]
)

adjusted_test_results["adjusted_prediction"] = (
    adjusted_test_results["base_prediction"]
    * np.maximum(1 + adjustment, 0)
)

# Validate predictions and compare metrics
assert len(adjusted_test_results) == 840
assert test_history["sales_date"].max() < test_start

actual = adjusted_test_results["total_units_sold"].to_numpy(
    dtype=float
)
assert actual.sum() > 0

comparison_records = []

for model_name, column in [
    ("Four-week weekday average", "base_prediction"),
    ("Calendar-adjusted average", "adjusted_prediction")
]:
    predicted = adjusted_test_results[column].to_numpy(dtype=float)
    assert np.isfinite(predicted).all()

    errors = predicted - actual

    comparison_records.append({
        "Model": model_name,
        "MAE": np.abs(errors).mean(),
        "RMSE": np.sqrt(np.mean(errors ** 2)),
        "WAPE (%)": 100 * np.abs(errors).sum() / actual.sum(),
        "Bias (%)": 100 * errors.sum() / actual.sum()
    })

test_calendar_comparison = pd.DataFrame(comparison_records)

print(test_calendar_comparison.round(2).to_string(index=False))

                    Model    MAE   RMSE  WAPE (%)  Bias (%)
Four-week weekday average 124.09 183.20     12.52     -4.99
Calendar-adjusted average 108.63 155.21     10.96     -3.30


In [27]:
# Save the final-period comparison and prediction details
test_calendar_comparison.to_csv(
    OUTPUT_PATH / "calendar_adjusted_test_metrics.csv",
    index=False
)

adjusted_test_results.to_csv(
    OUTPUT_PATH / "calendar_adjusted_test_predictions.csv",
    index=False
)

print("Final-period evaluation results saved.")

Final-period evaluation results saved.


In [28]:
from sklearn.base import clone

# Use all observed sales history
final_history = df_enriched.sort_values(
    ["store_id", "dept_id", "sales_date"]
).copy()

forecast_origin = final_history["sales_date"].max()

# Build historical baselines using past sales only
grouped_sales = final_history.groupby(
    ["store_id", "dept_id"]
)["total_units_sold"]

lag_values = pd.concat(
    [grouped_sales.shift(lag) for lag in [7, 14, 21, 28]],
    axis=1
)

final_history["historical_baseline"] = lag_values.mean(
    axis=1, skipna=False
)

final_fitting_data = final_history.dropna(
    subset=["historical_baseline"]
).copy()

final_fitting_data["relative_deviation"] = (
    final_fitting_data["total_units_sold"]
    / final_fitting_data["historical_baseline"].clip(lower=1)
    - 1
)

# Create state-specific SNAP features
for state in ["CA", "TX", "WI"]:
    final_fitting_data[f"snap_flag_{state}"] = (
        final_fitting_data["state_id"].eq(state)
        & final_fitting_data["snap_eligible"].eq(1)
    ).astype(int)

# Refit the selected adjustment model on all observed history
final_adjustment_model = clone(calendar_adjustment_model)

assert np.isfinite(
    final_fitting_data["relative_deviation"]
).all()

final_adjustment_model.fit(
    final_fitting_data[adjustment_columns],
    final_fitting_data["relative_deviation"]
)

print("Forecast origin:", forecast_origin.date())
print("Training rows:", len(final_fitting_data))
print("Final calendar adjustment model fitted.")

Forecast origin: 2016-05-22
Training rows: 21060
Final calendar adjustment model fitted.


In [29]:
# Calculate weekday averages from the latest four observed weeks
latest_history = final_history.loc[
    final_history["sales_date"]
    >= forecast_origin - pd.Timedelta(days=27)
].copy()

latest_history["day_of_week"] = (
    latest_history["sales_date"].dt.dayofweek
)

future_baselines = (
    latest_history.groupby(
        ["store_id", "dept_id", "day_of_week"],
        as_index=False
    )["total_units_sold"]
    .mean()
    .rename(columns={"total_units_sold": "baseline_units"})
)

# Create the future date and series combinations
forecast_dates = pd.DataFrame({
    "sales_date": pd.date_range(
        forecast_origin + pd.Timedelta(days=1),
        periods=28
    )
})

series = final_history[
    ["state_id", "store_id", "dept_id"]
].drop_duplicates()

calendar_adjusted_forecast = series.merge(
    forecast_dates, how="cross"
)

# Attach known future calendar information
calendar_adjusted_forecast = calendar_adjusted_forecast.merge(
    calendar_features[
        [
            "sales_date",
            "event_type_1",
            "event_type_2",
            "snap_CA",
            "snap_TX",
            "snap_WI"
        ]
    ],
    on="sales_date",
    how="left",
    validate="many_to_one"
)

# Validate calendar coverage before filling missing event labels
snap_columns = ["snap_CA", "snap_TX", "snap_WI"]
assert calendar_adjusted_forecast[snap_columns].notna().all().all()

for column in ["event_type_1", "event_type_2"]:
    calendar_adjusted_forecast[column] = (
        calendar_adjusted_forecast[column].fillna("None")
    )

for state in ["CA", "TX", "WI"]:
    calendar_adjusted_forecast[f"snap_flag_{state}"] = (
        calendar_adjusted_forecast["state_id"].eq(state)
        & calendar_adjusted_forecast[f"snap_{state}"].eq(1)
    ).astype(int)

# Apply the weekday baseline and calendar adjustment
calendar_adjusted_forecast["day_of_week"] = (
    calendar_adjusted_forecast["sales_date"].dt.dayofweek
)

calendar_adjusted_forecast = calendar_adjusted_forecast.merge(
    future_baselines,
    on=["store_id", "dept_id", "day_of_week"],
    how="left",
    validate="many_to_one"
)

adjustment = final_adjustment_model.predict(
    calendar_adjusted_forecast[adjustment_columns]
)

calendar_adjusted_forecast["adjustment_factor"] = np.maximum(
    1 + adjustment, 0
)

calendar_adjusted_forecast["predicted_units"] = (
    calendar_adjusted_forecast["baseline_units"]
    * calendar_adjusted_forecast["adjustment_factor"]
)

calendar_adjusted_forecast["forecast_origin"] = forecast_origin
calendar_adjusted_forecast["model"] = "Calendar-adjusted weekday average"

calendar_adjusted_forecast = calendar_adjusted_forecast.sort_values(
    ["store_id", "dept_id", "sales_date"]
).reset_index(drop=True)

# Check forecast completeness and validity
assert len(calendar_adjusted_forecast) == 840
assert not calendar_adjusted_forecast.duplicated(
    ["store_id", "dept_id", "sales_date"]
).any()
assert np.isfinite(
    calendar_adjusted_forecast["predicted_units"]
).all()
assert calendar_adjusted_forecast["predicted_units"].ge(0).all()

print("Forecast rows:", len(calendar_adjusted_forecast))
print(
    "Date range:",
    calendar_adjusted_forecast["sales_date"].min().date(),
    "to",
    calendar_adjusted_forecast["sales_date"].max().date()
)
print(
    "Total predicted units:",
    round(calendar_adjusted_forecast["predicted_units"].sum())
)

display(calendar_adjusted_forecast[
    ["sales_date", "store_id", "dept_id", "baseline_units", "predicted_units"]
].head(10))

Forecast rows: 840
Date range: 2016-05-23 to 2016-06-19
Total predicted units: 847173


,sales_date,store_id,dept_id,baseline_units,predicted_units
0,2016-05-23,CA_1,FOODS_1,282.75,273.643623
1,2016-05-24,CA_1,FOODS_1,304.75,294.935081
2,2016-05-25,CA_1,FOODS_1,304.00,294.209235
3,2016-05-26,CA_1,FOODS_1,306.25,296.386771
4,2016-05-27,CA_1,FOODS_1,383.25,370.906873
5,2016-05-28,CA_1,FOODS_1,447.75,433.329557
6,2016-05-29,CA_1,FOODS_1,368.00,356.148022
7,2016-05-30,CA_1,FOODS_1,282.75,248.646459
8,2016-05-31,CA_1,FOODS_1,304.75,294.935081
9,2016-06-01,CA_1,FOODS_1,304.00,326.998511


In [30]:
# Select forecast fields for reporting
forecast_export = calendar_adjusted_forecast[
    [
        "forecast_origin",
        "sales_date",
        "state_id",
        "store_id",
        "dept_id",
        "baseline_units",
        "adjustment_factor",
        "predicted_units",
        "model"
    ]
].copy()

# Save the calendar-adjusted forecast as a separate file
forecast_export.to_csv(
    OUTPUT_PATH / "future_sales_forecast_calendar_adjusted.csv",
    index=False,
    date_format="%Y-%m-%d"
)

print("Forecast saved.")
print("Rows:", len(forecast_export))
print("Stores:", forecast_export["store_id"].nunique())
print("Departments:", forecast_export["dept_id"].nunique())
print("Forecast days:", forecast_export["sales_date"].nunique())

Forecast saved.
Rows: 840
Stores: 10
Departments: 3
Forecast days: 28


In [31]:
# Prepare historical forecast deviations for anomaly analysis
anomaly_review = adjusted_test_results[
    [
        "sales_date",
        "state_id",
        "store_id",
        "dept_id",
        "total_units_sold",
        "adjusted_prediction",
        "event_type_1",
        "event_type_2",
        "snap_eligible"
    ]
].copy()

anomaly_review = anomaly_review.rename(columns={
    "total_units_sold": "actual_units",
    "adjusted_prediction": "expected_units"
})

# Positive deviations indicate sales above the forecast
anomaly_review["deviation_units"] = (
    anomaly_review["actual_units"]
    - anomaly_review["expected_units"]
)

# Avoid percentage calculations when the forecast is zero
anomaly_review["deviation_pct"] = (
    100
    * anomaly_review["deviation_units"]
    / anomaly_review["expected_units"].replace(0, np.nan)
)

anomaly_review["absolute_deviation_units"] = (
    anomaly_review["deviation_units"].abs()
)

assert len(anomaly_review) == 840
assert anomaly_review[
    ["actual_units", "expected_units", "deviation_units"]
].notna().all().all()

display(
    anomaly_review.sort_values(
        "absolute_deviation_units",
        ascending=False
    )[
        [
            "sales_date",
            "store_id",
            "dept_id",
            "actual_units",
            "expected_units",
            "deviation_units",
            "deviation_pct"
        ]
    ].head(10).round(2)
)

,sales_date,store_id,dept_id,actual_units,expected_units,deviation_units,deviation_pct
237,2016-05-08,CA_3,FOODS_3,3001,3914.10,-913.10,-23.33
742,2016-05-09,WI_2,FOODS_3,3388,2597.33,790.67,30.44
657,2016-05-08,WI_1,FOODS_3,1919,2618.68,-699.68,-26.72
76,2016-05-15,CA_1,FOODS_3,3521,2935.57,585.43,19.94
230,2016-05-01,CA_3,FOODS_3,3359,3941.05,-582.05,-14.77
744,2016-05-11,WI_2,FOODS_3,3103,2521.65,581.35,23.05
153,2016-05-08,CA_2,FOODS_3,2222,2793.58,-571.58,-20.46
733,2016-04-30,WI_2,FOODS_3,2122,2651.38,-529.38,-19.97
825,2016-05-08,WI_3,FOODS_3,2384,2911.42,-527.42,-18.12
726,2016-05-21,WI_2,FOODS_2,850,1373.73,-523.73,-38.12


In [32]:
# Define initial business alert thresholds
percentage_threshold = 25
units_threshold = 100

anomaly_review["is_alert"] = (
    anomaly_review["deviation_pct"].abs().ge(percentage_threshold)
    & anomaly_review["absolute_deviation_units"].ge(units_threshold)
)

anomaly_review["alert_type"] = np.select(
    [
        anomaly_review["is_alert"]
        & anomaly_review["deviation_units"].lt(0),

        anomaly_review["is_alert"]
        & anomaly_review["deviation_units"].gt(0)
    ],
    ["Below expected", "Above expected"],
    default="Within threshold"
)

alerts = anomaly_review.loc[
    anomaly_review["is_alert"]
].sort_values(
    "absolute_deviation_units",
    ascending=False
).copy()

print("Records reviewed:", len(anomaly_review))
print("Alerts:", len(alerts))
print(
    "Alert rate:",
    f"{anomaly_review['is_alert'].mean():.1%}"
)

print(alerts["alert_type"].value_counts())

display(
    alerts[
        [
            "sales_date",
            "store_id",
            "dept_id",
            "actual_units",
            "expected_units",
            "deviation_units",
            "deviation_pct",
            "alert_type"
        ]
    ].head(10).round(2)
)

Records reviewed: 840
Alerts: 102
Alert rate: 12.1%
alert_type
Above expected    85
Below expected    17
Name: count, dtype: int64


,sales_date,store_id,dept_id,actual_units,expected_units,deviation_units,deviation_pct,alert_type
742,2016-05-09,WI_2,FOODS_3,3388,2597.33,790.67,30.44,Above expected
657,2016-05-08,WI_1,FOODS_3,1919,2618.68,-699.68,-26.72,Below expected
726,2016-05-21,WI_2,FOODS_2,850,1373.73,-523.73,-38.12,Below expected
720,2016-05-15,WI_2,FOODS_2,2011,1536.89,474.11,30.85,Above expected
716,2016-05-11,WI_2,FOODS_2,1723,1250.84,472.16,37.75,Above expected
705,2016-04-30,WI_2,FOODS_2,886,1342.68,-456.68,-34.01,Below expected
714,2016-05-09,WI_2,FOODS_2,1826,1378.07,447.93,32.50,Above expected
725,2016-05-20,WI_2,FOODS_2,815,1237.54,-422.54,-34.14,Below expected
727,2016-05-22,WI_2,FOODS_2,836,1245.51,-409.51,-32.88,Below expected
700,2016-04-25,WI_2,FOODS_2,744,1116.81,-372.81,-33.38,Below expected


In [33]:
# Record the alert rules for traceability
anomaly_review["percentage_threshold"] = percentage_threshold
anomaly_review["units_threshold"] = units_threshold

# Save all reviewed records for alert-rate calculations
anomaly_review.to_csv(
    OUTPUT_PATH / "sales_anomaly_review.csv",
    index=False,
    date_format="%Y-%m-%d"
)

# Save flagged records for the alert detail table
anomaly_review.loc[anomaly_review["is_alert"]].to_csv(
    OUTPUT_PATH / "sales_alerts.csv",
    index=False,
    date_format="%Y-%m-%d"
)

print("Records reviewed:", len(anomaly_review))
print("Alerts:", int(anomaly_review["is_alert"].sum()))
print("Alert rate:", f"{anomaly_review['is_alert'].mean():.1%}")
print("Anomaly review and alerts saved.")

Records reviewed: 840
Alerts: 102
Alert rate: 12.1%
Anomaly review and alerts saved.


In [34]:
# Inspect calendar information for the shared low-sales date
review_date = pd.Timestamp("2016-05-08")

display(
    calendar_features.loc[
        calendar_features["sales_date"].eq(review_date)
    ]
)

# Compare all stores for the same department and date
display(
    anomaly_review.loc[
        anomaly_review["sales_date"].eq(review_date)
        & anomaly_review["dept_id"].eq("FOODS_3"),
        [
            "store_id",
            "actual_units",
            "expected_units",
            "deviation_units",
            "deviation_pct",
            "snap_eligible"
        ]
    ].sort_values("deviation_pct").round(2)
)

,sales_date,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI
1926,2016-05-08,Mother's day,Cultural,<NA>,<NA>,1,0,1


,store_id,actual_units,expected_units,deviation_units,deviation_pct,snap_eligible
657,WI_1,1919,2618.68,-699.68,-26.72,1
237,CA_3,3001,3914.10,-913.10,-23.33,1
153,CA_2,2222,2793.58,-571.58,-20.46,1
321,CA_4,1161,1439.36,-278.36,-19.34,1
825,WI_3,2384,2911.42,-527.42,-18.12,1
405,TX_1,1494,1800.65,-306.65,-17.03,0
489,TX_2,1937,2323.65,-386.65,-16.64,0
69,CA_1,2771,3174.57,-403.57,-12.71,1
741,WI_2,2827,3137.00,-310.00,-9.88,1
573,TX_3,2036,1888.48,147.52,7.81,0


In [35]:
# Find Mother's Day dates within the observed sales period
mother_days = calendar_features.loc[
    calendar_features["event_name_1"]
    .str.contains("mother", case=False, na=False),
    "sales_date"
]

mother_days = mother_days.loc[
    mother_days.between(
        df_enriched["sales_date"].min(),
        df_enriched["sales_date"].max()
    )
]

holiday_comparisons = []

for holiday_date in mother_days:
    # Select the same weekday from the previous four weeks
    reference_dates = [
        holiday_date - pd.Timedelta(days=lag)
        for lag in [7, 14, 21, 28]
    ]

    reference = (
        df_enriched.loc[
            df_enriched["sales_date"].isin(reference_dates)
            & df_enriched["dept_id"].eq("FOODS_3")
        ]
        .groupby("store_id", as_index=False)
        .agg(
            previous_weekday_avg=("total_units_sold", "mean"),
            reference_days=("sales_date", "nunique")
        )
    )

    actual = df_enriched.loc[
        df_enriched["sales_date"].eq(holiday_date)
        & df_enriched["dept_id"].eq("FOODS_3"),
        ["sales_date", "store_id", "total_units_sold"]
    ]

    comparison = actual.merge(
        reference, on="store_id", validate="one_to_one"
    )

    assert len(comparison) == 10
    assert comparison["reference_days"].eq(4).all()

    comparison["change_pct"] = (
        100 * (
            comparison["total_units_sold"]
            / comparison["previous_weekday_avg"].replace(0, np.nan)
            - 1
        )
    )

    holiday_comparisons.append(comparison)

mother_day_review = pd.concat(
    holiday_comparisons, ignore_index=True
)

display(mother_day_review.round(2))

,sales_date,store_id,total_units_sold,previous_weekday_avg,reference_days,change_pct
0,2015-05-10,CA_1,2542,2958.25,4,-14.07
1,2015-05-10,CA_2,883,1153.25,4,-23.43
2,2015-05-10,CA_3,3070,3582.25,4,-14.30
3,2015-05-10,CA_4,1109,1242.25,4,-10.73
4,2015-05-10,TX_1,1652,1849.00,4,-10.65
5,2015-05-10,TX_2,1831,2245.25,4,-18.45
6,2015-05-10,TX_3,1989,2030.25,4,-2.03
7,2015-05-10,WI_1,1477,1984.25,4,-25.56
8,2015-05-10,WI_2,1602,2248.00,4,-28.74
9,2015-05-10,WI_3,1603,2121.50,4,-24.44


In [36]:
# Review daily sales across departments at WI_2
store_review = df_enriched.loc[
    df_enriched["store_id"].eq("WI_2")
    & df_enriched["sales_date"].between(
        "2016-05-14", "2016-05-22"
    )
].copy()

display(
    store_review.pivot(
        index="sales_date",
        columns="dept_id",
        values="total_units_sold"
    )
)

# Inspect forecast deviations for FOODS_2
display(
    anomaly_review.loc[
        anomaly_review["store_id"].eq("WI_2")
        & anomaly_review["dept_id"].eq("FOODS_2")
        & anomaly_review["sales_date"].between(
            "2016-05-14", "2016-05-22"
        ),
        [
            "sales_date",
            "actual_units",
            "expected_units",
            "deviation_pct",
            "event_type_1",
            "snap_eligible"
        ]
    ].round(2)
)

dept_id,FOODS_1,FOODS_2,FOODS_3
sales_date,,,
2016-05-14,578,1934,3460
2016-05-15,574,2011,3593
2016-05-16,454,1203,2222
2016-05-17,386,1023,2028
2016-05-18,394,924,1811
2016-05-19,336,800,2062
2016-05-20,433,815,2090
2016-05-21,427,850,2330
2016-05-22,398,836,2323


,sales_date,actual_units,expected_units,deviation_pct,event_type_1,snap_eligible
719,2016-05-14,1934,1695.11,14.09,None,1
720,2016-05-15,2011,1536.89,30.85,None,1
721,2016-05-16,1203,1116.81,7.72,None,0
722,2016-05-17,1023,1198.42,-14.64,None,0
723,2016-05-18,924,1013.70,-8.85,None,0
724,2016-05-19,800,1015.63,-21.23,None,0
725,2016-05-20,815,1237.54,-34.14,None,0
726,2016-05-21,850,1373.73,-38.12,None,0
727,2016-05-22,836,1245.51,-32.88,None,0


In [37]:
# Compare forecast deviations across all departments at WI_2
wi2_department_review = anomaly_review.loc[
    anomaly_review["store_id"].eq("WI_2")
    & anomaly_review["sales_date"].between(
        "2016-05-14", "2016-05-22"
    )
]

display(
    wi2_department_review.pivot(
        index="sales_date",
        columns="dept_id",
        values="deviation_pct"
    ).round(2)
)

dept_id,FOODS_1,FOODS_2,FOODS_3
sales_date,,,
2016-05-14,-2.76,14.09,3.37
2016-05-15,21.01,30.85,11.75
2016-05-16,40.52,7.72,5.56
2016-05-17,7.07,-14.64,-3.87
2016-05-18,32.87,-8.85,-11.38
2016-05-19,18.32,-21.23,1.42
2016-05-20,3.41,-34.14,-16.92
2016-05-21,-11.36,-38.12,-14.11
2016-05-22,3.53,-32.88,-10.85


In [38]:
from pathlib import Path

# Read only the fields needed for the investigation
investigation_path = Path(
    r"D:\ai-fmcg-sales-insights\data\processed"
    r"\fmcg_sales_WI_2.csv.gz"
)

investigation_parts = []

for chunk in pd.read_csv(
    investigation_path,
    usecols=["date", "item_id", "dept_id", "units_sold", "sell_price"],
    parse_dates=["date"],
    chunksize=100_000
):
    selected = chunk.loc[
        chunk["dept_id"].eq("FOODS_2")
        & chunk["date"].between("2016-04-22", "2016-05-22")
    ].copy()

    investigation_parts.append(selected)

wi2_foods2_detail = pd.concat(
    investigation_parts, ignore_index=True
)

assert not wi2_foods2_detail.duplicated(
    ["date", "item_id"]
).any()

# Summarize daily sales and price coverage
daily_investigation = (
    wi2_foods2_detail.groupby("date")
    .agg(
        units_sold=("units_sold", "sum"),
        product_count=("item_id", "nunique"),
        zero_sales_products=("units_sold", lambda x: x.eq(0).sum()),
        missing_price_products=("sell_price", lambda x: x.isna().sum())
    )
)

display(daily_investigation.loc["2016-05-14":"2016-05-22"])

,units_sold,product_count,zero_sales_products,missing_price_products
date,,,,
2016-05-14,1934,398,117,0
2016-05-15,2011,398,100,0
2016-05-16,1203,398,139,0
2016-05-17,1023,398,154,0
2016-05-18,924,398,175,0
2016-05-19,800,398,176,0
2016-05-20,815,398,153,0
2016-05-21,850,398,140,0
2016-05-22,836,398,158,0


In [39]:
# Compare the three alert days with matching weekdays in prior weeks
alert_dates = pd.date_range("2016-05-20", "2016-05-22")

reference_dates = [
    date - pd.Timedelta(days=lag)
    for date in alert_dates
    for lag in [7, 14, 21, 28]
]

reference = (
    wi2_foods2_detail.loc[
        wi2_foods2_detail["date"].isin(reference_dates)
    ]
    .groupby("item_id")
    .agg(
        reference_units=("units_sold", "sum"),
        reference_days=("date", "nunique"),
        reference_price=("sell_price", "mean")
    )
)

assert reference["reference_days"].eq(12).all()

# Convert four comparison weekends into one three-day baseline
reference["baseline_3day_units"] = reference["reference_units"] / 4

actual = (
    wi2_foods2_detail.loc[
        wi2_foods2_detail["date"].isin(alert_dates)
    ]
    .groupby("item_id")
    .agg(
        actual_3day_units=("units_sold", "sum"),
        actual_days=("date", "nunique"),
        current_price=("sell_price", "mean")
    )
)

assert actual["actual_days"].eq(3).all()

item_decline_review = reference.join(actual, how="inner")
assert len(item_decline_review) == 398

item_decline_review["units_change"] = (
    item_decline_review["actual_3day_units"]
    - item_decline_review["baseline_3day_units"]
)

display(
    item_decline_review[
        [
            "baseline_3day_units",
            "actual_3day_units",
            "units_change",
            "reference_price",
            "current_price"
        ]
    ].sort_values("units_change").head(15).round(2)
)

,baseline_3day_units,actual_3day_units,units_change,reference_price,current_price
item_id,,,,,
FOODS_2_360,178.50,94,-84.50,0.94,0.94
FOODS_2_013,151.50,98,-53.50,1.68,1.68
FOODS_2_222,63.75,11,-52.75,2.50,2.50
FOODS_2_128,85.25,49,-36.25,0.94,0.94
FOODS_2_204,43.25,12,-31.25,1.00,1.00
FOODS_2_021,53.00,22,-31.00,0.94,0.94
FOODS_2_016,44.00,13,-31.00,1.25,1.25
FOODS_2_349,34.25,11,-23.25,5.98,5.98
FOODS_2_282,30.00,9,-21.00,6.72,6.72


In [40]:
# Measure how broadly sales declined across products
declining_items = item_decline_review.loc[
    item_decline_review["units_change"] < 0
].copy()

gross_decline = -declining_items["units_change"].sum()
top10_decline = -declining_items.nsmallest(
    10, "units_change"
)["units_change"].sum()

baseline_total = item_decline_review["baseline_3day_units"].sum()
actual_total = item_decline_review["actual_3day_units"].sum()

print("Products reviewed:", len(item_decline_review))
print("Products with lower sales:", len(declining_items))
print(
    "Share of products with lower sales:",
    f"{len(declining_items) / len(item_decline_review):.1%}"
)
print("Historical three-day baseline:", round(baseline_total, 2))
print("Actual three-day sales:", round(actual_total, 2))
print(
    "Change versus historical baseline:",
    f"{100 * (actual_total / baseline_total - 1):.2f}%"
)
print(
    "Top 10 share of gross sales declines:",
    f"{top10_decline / gross_decline:.1%}"
    if gross_decline > 0 else "N/A"
)

Products reviewed: 398
Products with lower sales: 271
Share of products with lower sales: 68.1%
Historical three-day baseline: 3650.25
Actual three-day sales: 2501
Change versus historical baseline: -31.48%
Top 10 share of gross sales declines: 27.5%


In [41]:
# Compare SNAP and event context for actual and reference dates
context_columns = [
    "sales_date",
    "snap_WI",
    "event_name_1",
    "event_name_2"
]

current_context = calendar_features.loc[
    calendar_features["sales_date"].isin(alert_dates),
    context_columns
].copy()

reference_context = calendar_features.loc[
    calendar_features["sales_date"].isin(reference_dates),
    context_columns
].copy()

assert len(current_context) == 3
assert len(reference_context) == 12

print("Current three days:")
display(current_context)

print("Previous four matching weekends:")
display(reference_context.sort_values("sales_date"))

print("Current SNAP-flag days:", int(current_context["snap_WI"].sum()))
print("Reference SNAP-flag days:", int(reference_context["snap_WI"].sum()))

Current three days:


,sales_date,snap_WI,event_name_1,event_name_2
1938,2016-05-20,0,<NA>,<NA>
1939,2016-05-21,0,<NA>,<NA>
1940,2016-05-22,0,<NA>,<NA>


Previous four matching weekends:


,sales_date,snap_WI,event_name_1,event_name_2
1910,2016-04-22,0,<NA>,<NA>
1911,2016-04-23,0,<NA>,<NA>
1912,2016-04-24,0,<NA>,<NA>
1917,2016-04-29,0,<NA>,<NA>
1918,2016-04-30,0,Pesach End,<NA>
1919,2016-05-01,0,OrthodoxEaster,<NA>
1924,2016-05-06,1,<NA>,<NA>
1925,2016-05-07,0,<NA>,<NA>
1926,2016-05-08,1,Mother's day,<NA>
1931,2016-05-13,0,<NA>,<NA>


Current SNAP-flag days: 0
Reference SNAP-flag days: 4


In [42]:
# Attach calendar context to product-level history
matched_history = wi2_foods2_detail.merge(
    calendar_features[
        ["sales_date", "snap_WI", "event_name_1", "event_name_2"]
    ],
    left_on="date",
    right_on="sales_date",
    how="left",
    validate="many_to_one"
)

matched_history["day_of_week"] = matched_history["date"].dt.dayofweek

# Select comparable dates strictly before the alert period
comparable = matched_history.loc[
    matched_history["date"].lt(pd.Timestamp("2016-05-20"))
    & matched_history["day_of_week"].isin([4, 5, 6])
    & matched_history["snap_WI"].eq(0)
    & matched_history["event_name_1"].fillna("None").eq("None")
    & matched_history["event_name_2"].fillna("None").eq("None")
]

# Calculate department totals before averaging each weekday
comparable_daily = (
    comparable.groupby(["date", "day_of_week"], as_index=False)
    ["units_sold"].sum()
)

matched_baseline = comparable_daily.groupby("day_of_week").agg(
    average_units=("units_sold", "mean"),
    reference_days=("date", "nunique")
)

assert set(matched_baseline.index) == {4, 5, 6}

matched_total = matched_baseline["average_units"].sum()
actual_total = wi2_foods2_detail.loc[
    wi2_foods2_detail["date"].isin(alert_dates),
    "units_sold"
].sum()

display(matched_baseline.round(2))

print("Matched three-day baseline:", round(matched_total, 2))
print("Actual three-day sales:", actual_total)
print(
    "Change versus matched baseline:",
    f"{100 * (actual_total / matched_total - 1):.2f}%"
)

,average_units,reference_days
day_of_week,,
4,1006.33,3
5,1111.00,2
6,932.00,1


Matched three-day baseline: 3049.33
Actual three-day sales: 2501
Change versus matched baseline: -17.98%


In [43]:
# Select the same three days across all stores
state_review = anomaly_review.loc[
    anomaly_review["sales_date"].between(
        "2016-05-20", "2016-05-22"
    )
].copy()

def summarize_state_deviation(data):
    summary = (
        data.groupby("state_id", as_index=False)
        .agg(
            actual_units=("actual_units", "sum"),
            expected_units=("expected_units", "sum"),
            stores_reviewed=("store_id", "nunique")
        )
    )

    summary["deviation_units"] = (
        summary["actual_units"] - summary["expected_units"]
    )

    summary["deviation_pct"] = (
        100 * summary["deviation_units"]
        / summary["expected_units"].replace(0, np.nan)
    )

    return summary.round(2)

print("All food departments: May 20–22")
display(summarize_state_deviation(state_review))

print("FOODS_2 only: May 20–22")
display(
    summarize_state_deviation(
        state_review.loc[state_review["dept_id"].eq("FOODS_2")]
    )
)

# Check whether the state result is driven by one store
store_comparison = (
    state_review.loc[state_review["dept_id"].eq("FOODS_2")]
    .groupby(["state_id", "store_id"], as_index=False)
    .agg(
        actual_units=("actual_units", "sum"),
        expected_units=("expected_units", "sum")
    )
)

store_comparison["deviation_pct"] = (
    100 * (
        store_comparison["actual_units"]
        / store_comparison["expected_units"].replace(0, np.nan)
        - 1
    )
)

print("FOODS_2 by store:")
display(
    store_comparison.sort_values(
        ["state_id", "deviation_pct"]
    ).round(2)
)

All food departments: May 20–22


,state_id,actual_units,expected_units,stores_reviewed,deviation_units,deviation_pct
0,CA,41813,39226.61,4,2586.39,6.59
1,TX,26299,23051.38,3,3247.62,14.09
2,WI,29590,32196.40,3,-2606.40,-8.10


FOODS_2 only: May 20–22


,state_id,actual_units,expected_units,stores_reviewed,deviation_units,deviation_pct
0,CA,7114,6978.05,4,135.95,1.95
1,TX,3953,3891.80,3,61.20,1.57
2,WI,6223,8182.27,3,-1959.27,-23.95


FOODS_2 by store:


,state_id,store_id,actual_units,expected_units,deviation_pct
2,CA,CA_3,2004,2217.44,-9.63
0,CA,CA_1,1780,1833.25,-2.90
3,CA,CA_4,1268,1147.96,10.46
1,CA,CA_2,2062,1779.41,15.88
4,TX,TX_1,1052,1128.16,-6.75
6,TX,TX_3,1537,1527.31,0.63
5,TX,TX_2,1364,1236.34,10.33
8,WI,WI_2,2501,3856.79,-35.15
7,WI,WI_1,1982,2348.55,-15.61
9,WI,WI_3,1740,1976.93,-11.98


In [44]:
# Compare the same department and dates across earlier months
monthly_review = df_enriched.loc[
    df_enriched["dept_id"].eq("FOODS_2")
    & df_enriched["sales_date"].between("2016-01-01", "2016-05-22")
].copy()

monthly_review["year_month"] = (
    monthly_review["sales_date"].dt.to_period("M").astype(str)
)

monthly_review["period"] = np.select(
    [
        monthly_review["sales_date"].dt.day.between(13, 15),
        monthly_review["sales_date"].dt.day.between(20, 22)
    ],
    ["Days 13–15", "Days 20–22"],
    default="Other"
)

monthly_comparison = (
    monthly_review.loc[monthly_review["period"].ne("Other")]
    .groupby(["year_month", "state_id", "period"])
    ["total_units_sold"].sum()
    .unstack("period")
)

monthly_comparison["change_pct"] = (
    100 * (
        monthly_comparison["Days 20–22"]
        / monthly_comparison["Days 13–15"]
        - 1
    )
)

display(monthly_comparison.round(2))

period               Days 13–15  Days 20–22  change_pct
year_month state_id                                    
2016-01    CA              4981        5091        2.21
           TX              4075        3321      -18.50
           WI              7807        5034      -35.52
2016-02    CA              6602        6955        5.35
           TX              5029        4147      -17.54
           WI              9234        6740      -27.01
2016-03    CA              6758        5552      -17.85
           TX              4665        3792      -18.71
           WI              8703        6331      -27.25
2016-04    CA              4977        5128        3.03
           TX              3761        3411       -9.31
           WI              7565        5615      -25.78
2016-05    CA              7634        7114       -6.81
           TX              4573        3953      -13.56
           WI             10267        6223      -39.39

In [45]:
# Create calendar features for an exploratory model upgrade
pattern_data = df_enriched.copy()

day_of_month = pattern_data["sales_date"].dt.day

pattern_data["month_phase"] = np.select(
    [
        day_of_month.le(7),
        day_of_month.le(15),
        day_of_month.le(22)
    ],
    ["Days_01_07", "Days_08_15", "Days_16_22"],
    default="Days_23_End"
)

# Allow monthly patterns to vary by state and department
pattern_data["state_month_phase"] = (
    pattern_data["state_id"]
    + "_"
    + pattern_data["month_phase"]
)

pattern_data["state_dept_month_phase"] = (
    pattern_data["state_id"]
    + "_"
    + pattern_data["dept_id"]
    + "_"
    + pattern_data["month_phase"]
)

# Allow SNAP associations to vary by state and department
pattern_data["state_dept_snap"] = (
    pattern_data["state_id"]
    + "_"
    + pattern_data["dept_id"]
    + "_SNAP_"
    + pattern_data["snap_eligible"].astype(int).astype(str)
)

pattern_feature_columns = [
    "state_month_phase",
    "state_dept_month_phase",
    "state_dept_snap"
]

assert len(pattern_data) == len(df_enriched)
assert pattern_data[pattern_feature_columns].notna().all().all()

display(
    pattern_data.loc[
        pattern_data["store_id"].eq("WI_2")
        & pattern_data["dept_id"].eq("FOODS_2")
        & pattern_data["sales_date"].between(
            "2016-05-14", "2016-05-22"
        ),
        [
            "sales_date",
            "month_phase",
            "state_dept_month_phase",
            "state_dept_snap"
        ]
    ]
)

,sales_date,month_phase,state_dept_month_phase,state_dept_snap
18971,2016-05-14,Days_08_15,WI_FOODS_2_Days_08_15,WI_FOODS_2_SNAP_1
18972,2016-05-15,Days_08_15,WI_FOODS_2_Days_08_15,WI_FOODS_2_SNAP_1
18973,2016-05-16,Days_16_22,WI_FOODS_2_Days_16_22,WI_FOODS_2_SNAP_0
18974,2016-05-17,Days_16_22,WI_FOODS_2_Days_16_22,WI_FOODS_2_SNAP_0
18975,2016-05-18,Days_16_22,WI_FOODS_2_Days_16_22,WI_FOODS_2_SNAP_0
18976,2016-05-19,Days_16_22,WI_FOODS_2_Days_16_22,WI_FOODS_2_SNAP_0
18977,2016-05-20,Days_16_22,WI_FOODS_2_Days_16_22,WI_FOODS_2_SNAP_0
18978,2016-05-21,Days_16_22,WI_FOODS_2_Days_16_22,WI_FOODS_2_SNAP_0
18979,2016-05-22,Days_16_22,WI_FOODS_2_Days_16_22,WI_FOODS_2_SNAP_0


In [46]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge

# Add pattern features to the original pre-validation training data
pattern_training = adjustment_train.merge(
    pattern_data[
        ["sales_date", "store_id", "dept_id"]
        + pattern_feature_columns
    ],
    on=["sales_date", "store_id", "dept_id"],
    how="left",
    validate="one_to_one"
)

pattern_categories = [
    "event_type_1",
    "event_type_2"
] + pattern_feature_columns

pattern_columns = pattern_categories + snap_adjustment_columns

# Keep the same regularization strength for the initial comparison
pattern_preprocessor = ColumnTransformer([
    (
        "categories",
        OneHotEncoder(handle_unknown="ignore"),
        pattern_categories
    ),
    ("snap", "passthrough", snap_adjustment_columns)
])

pattern_adjustment_model = Pipeline([
    ("preprocessor", pattern_preprocessor),
    ("regressor", Ridge(alpha=100, solver="lsqr"))
])

assert pattern_training["sales_date"].max() < validation_start
assert pattern_training[pattern_columns].notna().all().all()

pattern_adjustment_model.fit(
    pattern_training[pattern_columns],
    pattern_training["relative_deviation"]
)

# Attach pattern features to the existing validation baseline
pattern_validation_results = adjusted_validation.merge(
    pattern_data[
        ["sales_date", "store_id", "dept_id"]
        + pattern_feature_columns
    ],
    on=["sales_date", "store_id", "dept_id"],
    how="left",
    validate="one_to_one"
)

adjustment = pattern_adjustment_model.predict(
    pattern_validation_results[pattern_columns]
)

pattern_validation_results["pattern_prediction"] = (
    pattern_validation_results["weekday_average_prediction"]
    * np.maximum(1 + adjustment, 0)
)

# Evaluate against observed validation sales
actual = pattern_validation_results["total_units_sold"].to_numpy(
    dtype=float
)
predicted = pattern_validation_results["pattern_prediction"].to_numpy(
    dtype=float
)

assert len(pattern_validation_results) == 840
assert np.isfinite(predicted).all()

errors = predicted - actual

pattern_metrics = pd.DataFrame([{
    "Model": "Calendar + state-department monthly patterns",
    "MAE": np.abs(errors).mean(),
    "RMSE": np.sqrt(np.mean(errors ** 2)),
    "WAPE (%)": 100 * np.abs(errors).sum() / actual.sum(),
    "Bias (%)": 100 * errors.sum() / actual.sum()
}])

print(pattern_metrics.round(2).to_string(index=False))

                                       Model   MAE   RMSE  WAPE (%)  Bias (%)
Calendar + state-department monthly patterns 79.67 115.41      8.46      2.38


In [47]:
from sklearn.base import clone

pattern_backtest_records = []

for start in backtest_starts:
    # Use only observations available before each forecast window
    history = pattern_data.loc[
        pattern_data["sales_date"] < start
    ].sort_values(
        ["store_id", "dept_id", "sales_date"]
    ).copy()

    future = pattern_data.loc[
        pattern_data["sales_date"].between(
            start, start + pd.Timedelta(days=27)
        )
    ].copy()

    grouped_sales = history.groupby(
        ["store_id", "dept_id"]
    )["total_units_sold"]

    history["historical_baseline"] = pd.concat(
        [grouped_sales.shift(lag) for lag in [7, 14, 21, 28]],
        axis=1
    ).mean(axis=1, skipna=False)

    fitting = history.dropna(
        subset=["historical_baseline"]
    ).copy()

    fitting["relative_deviation"] = (
        fitting["total_units_sold"]
        / fitting["historical_baseline"].clip(lower=1)
        - 1
    )

    # Create the same SNAP features for training and forecasting
    for state in ["CA", "TX", "WI"]:
        for frame in [fitting, future]:
            frame[f"snap_flag_{state}"] = (
                frame["state_id"].eq(state)
                & frame["snap_eligible"].eq(1)
            ).astype(int)

    fold_model = clone(pattern_adjustment_model)
    fold_model.fit(
        fitting[pattern_columns],
        fitting["relative_deviation"]
    )

    # Keep the weekday baseline fixed throughout the 28-day horizon
    recent = history.loc[
        history["sales_date"] >= start - pd.Timedelta(days=28)
    ].copy()

    recent["day_of_week"] = recent["sales_date"].dt.dayofweek
    future["day_of_week"] = future["sales_date"].dt.dayofweek

    baseline = (
        recent.groupby(
            ["store_id", "dept_id", "day_of_week"],
            as_index=False
        )["total_units_sold"]
        .mean()
        .rename(columns={"total_units_sold": "base_prediction"})
    )

    future = future.merge(
        baseline,
        on=["store_id", "dept_id", "day_of_week"],
        how="left",
        validate="many_to_one"
    )

    predicted = (
        future["base_prediction"].to_numpy()
        * np.maximum(
            1 + fold_model.predict(future[pattern_columns]), 0
        )
    )

    actual = future["total_units_sold"].to_numpy(dtype=float)

    assert len(future) == 840
    assert np.isfinite(predicted).all()
    assert actual.sum() > 0

    errors = predicted - actual

    pattern_backtest_records.append({
        "Forecast Start": start.date(),
        "Forecast End": (start + pd.Timedelta(days=27)).date(),
        "Model": "Calendar + monthly patterns",
        "WAPE (%)": 100 * np.abs(errors).sum() / actual.sum(),
        "Bias (%)": 100 * errors.sum() / actual.sum()
    })

pattern_backtest_results = pd.DataFrame(pattern_backtest_records)

pattern_backtest_comparison = pd.concat(
    [backtest_results, pattern_backtest_results],
    ignore_index=True
).sort_values(["Forecast Start", "Model"])

print(
    pattern_backtest_comparison.round(2).to_string(index=False)
)

Forecast Start Forecast End                       Model  WAPE (%)  Bias (%)
    2016-01-04   2016-01-31 Calendar + monthly patterns     18.76    -11.39
    2016-01-04   2016-01-31   Calendar-adjusted average     18.66    -10.69
    2016-01-04   2016-01-31   Four-week weekday average     18.97    -10.71
    2016-02-01   2016-02-28 Calendar + monthly patterns     12.99     -5.30
    2016-02-01   2016-02-28   Calendar-adjusted average     13.04     -5.29
    2016-02-01   2016-02-28   Four-week weekday average     13.52     -6.59
    2016-02-29   2016-03-27 Calendar + monthly patterns     10.76      0.66
    2016-02-29   2016-03-27   Calendar-adjusted average     11.10      0.62
    2016-02-29   2016-03-27   Four-week weekday average     11.30     -1.30
    2016-03-28   2016-04-24 Calendar + monthly patterns      8.46      2.38
    2016-03-28   2016-04-24   Calendar-adjusted average      9.25      2.38
    2016-03-28   2016-04-24   Four-week weekday average     10.42     -0.00


In [48]:
# Save the exploratory model comparison
pattern_backtest_comparison.to_csv(
    OUTPUT_PATH / "forecast_monthly_pattern_backtests.csv",
    index=False
)

pattern_metrics.to_csv(
    OUTPUT_PATH / "forecast_monthly_pattern_validation_metrics.csv",
    index=False
)

print("Monthly pattern model evaluation saved.")

Monthly pattern model evaluation saved.


In [49]:
from sklearn.base import clone

# Attach pattern features to the existing pre-test training data
pattern_test_training = test_fitting_data.merge(
    pattern_data[
        ["sales_date", "store_id", "dept_id"]
        + pattern_feature_columns
    ],
    on=["sales_date", "store_id", "dept_id"],
    how="left",
    validate="one_to_one"
)

assert pattern_test_training["sales_date"].max() < test_start
assert pattern_test_training[pattern_columns].notna().all().all()

# Refit the upgraded model using pre-test history only
pattern_test_model = clone(pattern_adjustment_model)

pattern_test_model.fit(
    pattern_test_training[pattern_columns],
    pattern_test_training["relative_deviation"]
)

# Add pattern features to the existing test predictions
pattern_test_results = adjusted_test_results.merge(
    pattern_data[
        ["sales_date", "store_id", "dept_id"]
        + pattern_feature_columns
    ],
    on=["sales_date", "store_id", "dept_id"],
    how="left",
    validate="one_to_one"
)

adjustment = pattern_test_model.predict(
    pattern_test_results[pattern_columns]
)

pattern_test_results["pattern_prediction"] = (
    pattern_test_results["base_prediction"]
    * np.maximum(1 + adjustment, 0)
)

assert len(pattern_test_results) == 840
assert np.isfinite(
    pattern_test_results["pattern_prediction"]
).all()

# Compare overall errors across all three models
actual = pattern_test_results["total_units_sold"].to_numpy(dtype=float)

records = []

for model_name, column in [
    ("Four-week weekday average", "base_prediction"),
    ("Calendar-adjusted average", "adjusted_prediction"),
    ("Calendar + monthly patterns", "pattern_prediction")
]:
    predicted = pattern_test_results[column].to_numpy(dtype=float)
    errors = predicted - actual

    records.append({
        "Model": model_name,
        "WAPE (%)": 100 * np.abs(errors).sum() / actual.sum(),
        "Bias (%)": 100 * errors.sum() / actual.sum()
    })

pattern_test_comparison = pd.DataFrame(records)

print("Overall final-period comparison:")
print(pattern_test_comparison.round(2).to_string(index=False))

# Review the investigated WI FOODS_2 period separately
focus = pattern_test_results.loc[
    pattern_test_results["state_id"].eq("WI")
    & pattern_test_results["dept_id"].eq("FOODS_2")
    & pattern_test_results["sales_date"].between(
        "2016-05-20", "2016-05-22"
    )
]

focus_actual = focus["total_units_sold"].sum()

print("\nWI FOODS_2, May 20–22:")

for label, column in [
    ("Original calendar model", "adjusted_prediction"),
    ("Monthly pattern model", "pattern_prediction")
]:
    expected = focus[column].sum()
    deviation = 100 * (focus_actual / expected - 1)

    print(
        f"{label}: actual={focus_actual:.0f}, "
        f"expected={expected:.2f}, "
        f"deviation={deviation:.2f}%"
    )

Overall final-period comparison:
                      Model  WAPE (%)  Bias (%)
  Four-week weekday average     12.52     -4.99
  Calendar-adjusted average     10.96     -3.30
Calendar + monthly patterns     10.38     -3.13

WI FOODS_2, May 20–22:
Original calendar model: actual=6223, expected=8182.27, deviation=-23.95%
Monthly pattern model: actual=6223, expected=7860.21, deviation=-20.83%


In [50]:
# Save exploratory final-period metrics and predictions
pattern_test_comparison.to_csv(
    OUTPUT_PATH / "forecast_monthly_pattern_test_metrics.csv",
    index=False
)

pattern_test_results.to_csv(
    OUTPUT_PATH / "forecast_monthly_pattern_test_predictions.csv",
    index=False,
    date_format="%Y-%m-%d"
)

print("Monthly pattern evaluation saved.")

Monthly pattern evaluation saved.


In [51]:
from sklearn.base import clone

# Prepare all observed history for the selected model
final_pattern_training = pattern_data.sort_values(
    ["store_id", "dept_id", "sales_date"]
).copy()

forecast_origin = final_pattern_training["sales_date"].max()

# Build baselines using past sales only
grouped_sales = final_pattern_training.groupby(
    ["store_id", "dept_id"]
)["total_units_sold"]

final_pattern_training["historical_baseline"] = pd.concat(
    [grouped_sales.shift(lag) for lag in [7, 14, 21, 28]],
    axis=1
).mean(axis=1, skipna=False)

final_pattern_training = final_pattern_training.dropna(
    subset=["historical_baseline"]
).copy()

final_pattern_training["relative_deviation"] = (
    final_pattern_training["total_units_sold"]
    / final_pattern_training["historical_baseline"].clip(lower=1)
    - 1
)

# Create state-specific SNAP features
for state in ["CA", "TX", "WI"]:
    final_pattern_training[f"snap_flag_{state}"] = (
        final_pattern_training["state_id"].eq(state)
        & final_pattern_training["snap_eligible"].eq(1)
    ).astype(int)

# Validate inputs and refit the selected model
assert final_pattern_training[pattern_columns].notna().all().all()
assert np.isfinite(
    final_pattern_training["relative_deviation"]
).all()

final_pattern_model = clone(pattern_adjustment_model)

final_pattern_model.fit(
    final_pattern_training[pattern_columns],
    final_pattern_training["relative_deviation"]
)

print("Forecast origin:", forecast_origin.date())
print("Training rows:", len(final_pattern_training))
print("Final monthly pattern model fitted.")

Forecast origin: 2016-05-22
Training rows: 21060
Final monthly pattern model fitted.


In [52]:
# Reuse the future calendar and weekday baseline
monthly_pattern_forecast = calendar_adjusted_forecast.copy()

day_of_month = monthly_pattern_forecast["sales_date"].dt.day

monthly_pattern_forecast["month_phase"] = np.select(
    [
        day_of_month.le(7),
        day_of_month.le(15),
        day_of_month.le(22)
    ],
    ["Days_01_07", "Days_08_15", "Days_16_22"],
    default="Days_23_End"
)

# Create the same pattern features used during training
monthly_pattern_forecast["state_month_phase"] = (
    monthly_pattern_forecast["state_id"]
    + "_"
    + monthly_pattern_forecast["month_phase"]
)

monthly_pattern_forecast["state_dept_month_phase"] = (
    monthly_pattern_forecast["state_id"]
    + "_"
    + monthly_pattern_forecast["dept_id"]
    + "_"
    + monthly_pattern_forecast["month_phase"]
)

monthly_pattern_forecast["snap_eligible"] = 0

for state in ["CA", "TX", "WI"]:
    mask = monthly_pattern_forecast["state_id"].eq(state)

    monthly_pattern_forecast.loc[mask, "snap_eligible"] = (
        monthly_pattern_forecast.loc[mask, f"snap_{state}"]
    )

monthly_pattern_forecast["state_dept_snap"] = (
    monthly_pattern_forecast["state_id"]
    + "_"
    + monthly_pattern_forecast["dept_id"]
    + "_SNAP_"
    + monthly_pattern_forecast["snap_eligible"].astype(int).astype(str)
)

# Generate forecasts using the refitted monthly pattern model
assert monthly_pattern_forecast[pattern_columns].notna().all().all()

adjustment = final_pattern_model.predict(
    monthly_pattern_forecast[pattern_columns]
)

monthly_pattern_forecast["adjustment_factor"] = np.maximum(
    1 + adjustment, 0
)

monthly_pattern_forecast["predicted_units"] = (
    monthly_pattern_forecast["baseline_units"]
    * monthly_pattern_forecast["adjustment_factor"]
)

monthly_pattern_forecast["model"] = "Calendar + monthly patterns"
monthly_pattern_forecast["forecast_origin"] = forecast_origin

# Check forecast dates, unique keys, and valid predictions
assert len(monthly_pattern_forecast) == 840
assert monthly_pattern_forecast["sales_date"].min() == (
    forecast_origin + pd.Timedelta(days=1)
)
assert monthly_pattern_forecast["sales_date"].max() == (
    forecast_origin + pd.Timedelta(days=28)
)
assert not monthly_pattern_forecast.duplicated(
    ["sales_date", "store_id", "dept_id"]
).any()
assert np.isfinite(monthly_pattern_forecast["predicted_units"]).all()
assert monthly_pattern_forecast["predicted_units"].ge(0).all()

# Save reporting fields for Power BI
monthly_forecast_export = monthly_pattern_forecast[
    [
        "forecast_origin",
        "sales_date",
        "state_id",
        "store_id",
        "dept_id",
        "baseline_units",
        "adjustment_factor",
        "predicted_units",
        "model"
    ]
].copy()

monthly_forecast_export.to_csv(
    OUTPUT_PATH / "future_sales_forecast_monthly_patterns.csv",
    index=False,
    date_format="%Y-%m-%d"
)

print("Forecast saved.")
print("Rows:", len(monthly_forecast_export))
print(
    "Date range:",
    monthly_forecast_export["sales_date"].min().date(),
    "to",
    monthly_forecast_export["sales_date"].max().date()
)

display(monthly_forecast_export.head().round(2))

Forecast saved.
Rows: 840
Date range: 2016-05-23 to 2016-06-19


,forecast_origin,sales_date,state_id,store_id,dept_id,baseline_units,adjustment_factor,predicted_units,model
0,2016-05-22,2016-05-23,CA,CA_1,FOODS_1,282.75,0.99,280.28,Calendar + monthly patterns
1,2016-05-22,2016-05-24,CA,CA_1,FOODS_1,304.75,0.99,302.08,Calendar + monthly patterns
2,2016-05-22,2016-05-25,CA,CA_1,FOODS_1,304.00,0.99,301.34,Calendar + monthly patterns
3,2016-05-22,2016-05-26,CA,CA_1,FOODS_1,306.25,0.99,303.57,Calendar + monthly patterns
4,2016-05-22,2016-05-27,CA,CA_1,FOODS_1,383.25,0.99,379.90,Calendar + monthly patterns


In [53]:
# Use historical predictions made with pre-test training data
monthly_anomaly_review = pattern_test_results[
    [
        "sales_date",
        "state_id",
        "store_id",
        "dept_id",
        "total_units_sold",
        "pattern_prediction"
    ]
].copy()

monthly_anomaly_review = monthly_anomaly_review.rename(columns={
    "total_units_sold": "actual_units",
    "pattern_prediction": "expected_units"
})

monthly_anomaly_review["deviation_units"] = (
    monthly_anomaly_review["actual_units"]
    - monthly_anomaly_review["expected_units"]
)

monthly_anomaly_review["deviation_pct"] = (
    100 * monthly_anomaly_review["deviation_units"]
    / monthly_anomaly_review["expected_units"].replace(0, np.nan)
)

# Keep the same business alert thresholds
monthly_anomaly_review["is_alert"] = (
    monthly_anomaly_review["deviation_pct"].abs().ge(25)
    & monthly_anomaly_review["deviation_units"].abs().ge(100)
)

monthly_anomaly_review["alert_type"] = np.select(
    [
        monthly_anomaly_review["is_alert"]
        & monthly_anomaly_review["deviation_units"].lt(0),
        monthly_anomaly_review["is_alert"]
        & monthly_anomaly_review["deviation_units"].gt(0)
    ],
    ["Below expected", "Above expected"],
    default="Within threshold"
)

monthly_anomaly_review["model"] = "Calendar + monthly patterns"
monthly_anomaly_review["forecast_origin"] = (
    test_start - pd.Timedelta(days=1)
)
monthly_anomaly_review["percentage_threshold"] = 25
monthly_anomaly_review["units_threshold"] = 100

assert len(monthly_anomaly_review) == 840
assert np.isfinite(monthly_anomaly_review["expected_units"]).all()
assert not monthly_anomaly_review.duplicated(
    ["sales_date", "store_id", "dept_id"]
).any()

# Save all records so Power BI can calculate alert rates
monthly_anomaly_review.to_csv(
    OUTPUT_PATH / "sales_anomaly_review_monthly_patterns.csv",
    index=False,
    date_format="%Y-%m-%d"
)

print("Historical review saved.")
print("Records reviewed:", len(monthly_anomaly_review))
print("Alerts:", int(monthly_anomaly_review["is_alert"].sum()))
print("Alert rate:", f"{monthly_anomaly_review['is_alert'].mean():.1%}")

Historical review saved.
Records reviewed: 840
Alerts: 92
Alert rate: 11.0%
